In [ ]:
"""
LLM-HPO Training Pipeline v4.4
================================
Changes vs v4.3:
  BUG FIXES:
  1. Stage1 best_weights.ckpt was silently overwritten by Stage2
     → Now Stage1 saves to stage1_best.ckpt, Stage2 to stage2_best.ckpt
  2. load_best_weights used fragile .index file check
     → Now uses tf.train.latest_checkpoint which is reliable
  3. predicted_val_accuracy / predicted_val_loss missing from summary CSV
     → Required for Spearman rank-concordance analysis (core thesis metric)
  4. Requested_Method / Effective_Method missing from summary CSV
     → Needed to split results by prompt method in analysis
  5. logic text missing from summary CSV
     → Needed for judge scoring traceability
  6. No per-run config JSON backup
     → If CSV is lost or corrupted, run config is unrecoverable

  LOGGING ADDITIONS:
  7. Session-level session_log.csv: one row per run appended live with timestamp
  8. Per-run run_config.json: full metadata snapshot written before training starts
  9. Separate stage1/ and stage2/ subdirs keep all stage artifacts clean
"""

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import warnings
warnings.filterwarnings('ignore')

import gc, time, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, optimizers
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

tf.get_logger().setLevel('ERROR')

# ==========================================
# GPU SETUP
# ==========================================
tf.keras.mixed_precision.set_global_policy("mixed_float16")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

try:
    tf.config.optimizer.set_jit(False)
except Exception:
    pass

# ==========================================
# CONFIGURATION
# ==========================================
SESSION_TIME = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
DATASET_DIR  = "D:/WBC Research Files A7/Split/Split"
EXCEL_INPUT  = "E:/WBC Dataset 1/WBC Dataset/HPO_LLM_Analysis.xlsx"
EXCEL_SHEET  = "36_Selected_Combos"
RESULTS_DIR  = "E:/WBC Dataset 1/WBC Dataset/Results/LLM_HPO"
SESSION_DIR  = f"{RESULTS_DIR}/session_{SESSION_TIME}"
os.makedirs(SESSION_DIR, exist_ok=True)

# Output files
SUMMARY_CSV     = f"{SESSION_DIR}/llm_hpo_summary_{SESSION_TIME}.csv"
SESSION_LOG_CSV = f"{SESSION_DIR}/session_log_{SESSION_TIME}.csv"

# Training constants — match baseline exactly
IMG_SIZE      = (300, 300)
EPOCHS_STAGE1 = 3
EPOCHS_STAGE2 = 5
UNFREEZE_PCT  = 0.30    # matches old HPO search loop (not final-run 0.55)
SEED          = 42
DROPOUT_RATE  = 0.5
L2_REG        = 1e-5

tf.random.set_seed(SEED)
np.random.seed(SEED)


def aggressive_cleanup():
    try:
        keras.backend.clear_session()
        tf.keras.backend.clear_session()
    except Exception:
        pass
    gc.collect()


# ==========================================
# DATA LOADING
# ==========================================
def compute_class_weights(train_dir):
    classes = sorted(os.listdir(train_dir))
    counts  = [sum(len(f) for _, _, f in os.walk(os.path.join(train_dir, c)))
               for c in classes]
    total   = sum(counts)

    raw    = {c: total / (len(classes) * counts[i]) for i, c in enumerate(classes)}
    max_w  = max(raw.values())
    scaled = {c: min(10.0, max(0.0, raw[c] * (10.0 / max_w))) for c in classes}

    return {i: scaled[c] for i, c in enumerate(classes)}, len(classes), classes


def build_dataset(directory, batch_size, augment=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size=IMG_SIZE,
        batch_size=batch_size,
        seed=SEED,
        label_mode='int',
        shuffle=augment
    )

    aug_layer = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ], name="train_aug")

    def preprocess(x, y):
        x = tf.cast(x, tf.float32)
        if augment:
            x = aug_layer(x, training=True)
        x = tf.keras.applications.efficientnet.preprocess_input(x)
        return x, y

    return ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)


def load_all_splits(data_dir, batch_size):
    train_ds = build_dataset(os.path.join(data_dir, "train"), batch_size, augment=True)
    val_ds   = build_dataset(os.path.join(data_dir, "val"),   batch_size, augment=False)
    test_ds  = build_dataset(os.path.join(data_dir, "test"),  batch_size, augment=False)
    cw, n_cls, class_names = compute_class_weights(os.path.join(data_dir, "train"))
    return train_ds, val_ds, test_ds, cw, n_cls, class_names


# ==========================================
# MODEL
# ==========================================
def make_optimizer(lr):
    # Plain Adam with clipnorm — no amsgrad, no custom epsilon
    return optimizers.Adam(learning_rate=lr, clipnorm=1.0)


def build_model(num_classes, learning_rate):
    inputs     = layers.Input(shape=IMG_SIZE + (3,))
    base_model = EfficientNetB3(include_top=False, weights="imagenet", input_tensor=inputs)
    base_model.trainable = False

    x       = layers.GlobalAveragePooling2D(name="gap")(base_model.output)
    x       = layers.Dropout(DROPOUT_RATE, name="drop_head")(x)
    outputs = layers.Dense(
        num_classes, activation="softmax",
        kernel_regularizer=regularizers.l2(L2_REG),
        dtype="float32", name="classifier"
    )(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=make_optimizer(learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model, base_model


def unfreeze_top(model, base_model, stage1_lr):
    base_model.trainable = True
    total  = len(base_model.layers)
    freeze = int(total * (1 - UNFREEZE_PCT))

    for i, layer in enumerate(base_model.layers):
        layer.trainable = (
            (i >= freeze) and not isinstance(layer, layers.BatchNormalization)
        )

    model.compile(
        optimizer=make_optimizer(stage1_lr / 10.0),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


# ==========================================
# CALLBACKS
# FIX 1: stage1 and stage2 use separate checkpoint paths
# FIX 2: no ReduceLROnPlateau (would corrupt LLM-suggested LR)
# FIX 3: restore_best_weights=False, handled via load_best_weights()
# ==========================================
def get_callbacks(ckpt_path, log_csv_path):
    return [
        ModelCheckpoint(
            filepath=ckpt_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=0
        ),
        EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=False,  # handled by load_best_weights()
            verbose=0
        ),
        CSVLogger(log_csv_path, append=True)
    ]


def load_best_weights(model, ckpt_path):
    """
    FIX 2: use tf.train.latest_checkpoint for reliable weight restoration.
    Falls back silently if checkpoint doesn't exist (e.g. EarlyStopping at epoch 1).
    """
    ckpt_dir = os.path.dirname(ckpt_path)
    latest   = tf.train.latest_checkpoint(ckpt_dir)
    if latest:
        model.load_weights(latest)
    return model


# ==========================================
# EVALUATION & PLOTTING
# ==========================================
def evaluate_and_plot(model, test_ds, class_names, run_dir,
                      run_id, lr, bs, hist1, hist2, s1_actual):
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)

    y_true, y_pred, y_probs = [], [], []
    for x_batch, y_batch in test_ds:
        preds = model.predict(x_batch, verbose=0)
        y_true.extend(y_batch.numpy())
        y_pred.extend(np.argmax(preds, axis=1))
        y_probs.append(preds)

    y_true  = np.array(y_true)
    y_pred  = np.array(y_pred)
    y_probs = np.vstack(y_probs)

    # --- Raw predictions CSV ---
    prob_cols = [f"Prob_{c.replace(' ', '_')}" for c in class_names]
    df_preds  = pd.DataFrame(y_probs, columns=prob_cols)
    df_preds.insert(0, "True_Label_Idx",       y_true)
    df_preds.insert(1, "True_Label_Name",      [class_names[i] for i in y_true])
    df_preds.insert(2, "Predicted_Label_Idx",  y_pred)
    df_preds.insert(3, "Predicted_Label_Name", [class_names[i] for i in y_pred])
    df_preds.to_csv(f"{run_dir}/raw_test_predictions.csv", index=False)

    # --- Confusion matrix ---
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(
        figsize=(max(8, len(class_names)), max(6, len(class_names) - 1))
    )
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Run {run_id} | LR={lr:.6f} BS={bs} | Test Acc: {test_acc:.4f}")
    plt.tight_layout()
    plt.savefig(f"{run_dir}/confusion_matrix.png", dpi=150)
    plt.close()

    # --- Training curves ---
    acc      = hist1.history['accuracy']     + (hist2.history['accuracy']     if hist2 else [])
    val_acc  = hist1.history['val_accuracy'] + (hist2.history['val_accuracy'] if hist2 else [])
    loss     = hist1.history['loss']         + (hist2.history['loss']         if hist2 else [])
    val_loss = hist1.history['val_loss']     + (hist2.history['val_loss']     if hist2 else [])
    ep_range = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(ep_range, acc,     'b-', label='Train', linewidth=2)
    ax1.plot(ep_range, val_acc, 'r-', label='Val',   linewidth=2)
    ax1.axvline(s1_actual, color='k', linestyle='--', alpha=0.6, label='Unfreeze')
    ax1.set_title('Accuracy'); ax1.legend()

    ax2.plot(ep_range, loss,     'b-', label='Train', linewidth=2)
    ax2.plot(ep_range, val_loss, 'r-', label='Val',   linewidth=2)
    ax2.axvline(s1_actual, color='k', linestyle='--', alpha=0.6, label='Unfreeze')
    ax2.set_title('Loss'); ax2.legend()

    plt.tight_layout()
    plt.savefig(f"{run_dir}/training_curves.png", dpi=150)
    plt.close()

    # --- Classification report JSON ---
    report = classification_report(
        y_true, y_pred, target_names=class_names,
        output_dict=True, zero_division=0
    )
    with open(f"{run_dir}/classification_report.json", "w") as f:
        json.dump(report, f, indent=2)

    # --- Full history JSON ---
    def to_float_list(d):
        return {k: [float(v) for v in vals] for k, vals in d.items()}

    with open(f"{run_dir}/history_dict.json", "w") as f:
        json.dump({
            "stage1": to_float_list(hist1.history),
            "stage2": to_float_list(hist2.history) if hist2 else {}
        }, f, indent=2)

    return {
        "test_accuracy": round(test_acc,  4),
        "test_loss":     round(test_loss, 4),
        "macro_f1":      round(report["macro avg"]["f1-score"],    4),
        "weighted_f1":   round(report["weighted avg"]["f1-score"], 4),
        "best_val_loss": round(min(val_loss), 4),
        "best_val_acc":  round(max(val_acc),  4),
    }


# ==========================================
# SESSION LOG HELPER
# ==========================================
def append_session_log(row_dict):
    """Appends one row to the live session log CSV."""
    df = pd.DataFrame([row_dict])
    write_header = not os.path.exists(SESSION_LOG_CSV)
    df.to_csv(SESSION_LOG_CSV, mode='a', header=write_header, index=False)


# ==========================================
# MAIN EXPERIMENT LOOP
# ==========================================
def run_all_experiments():
    df_hpm = pd.read_excel(EXCEL_INPUT, sheet_name=EXCEL_SHEET, header=1)
    summary_rows = []

    print(f"\n🚀 Starting {len(df_hpm)} LLM-HPO runs  |  Session: {SESSION_TIME}")
    print(f"   UNFREEZE_PCT={UNFREEZE_PCT} | amsgrad=OFF | ReduceLR=OFF\n")

    for idx, row in df_hpm.iterrows():

        # --- Read all metadata from Excel row ---
        run_id            = row.get('#',                  idx + 1)
        lr                = float(row['Learning Rate'])
        bs                = int(row['Batch Size'])
        model_name        = str(row.get('Model',              ''))
        rag_type          = str(row.get('RAG Type',           ''))
        requested_method  = str(row.get('Requested_Method',   ''))
        effective_method  = str(row.get('Effective_Method',   ''))
        predicted_val_acc = row.get('predicted_val_accuracy', '')
        predicted_val_loss= row.get('predicted_val_loss',     '')
        logic_text        = str(row.get('logic',              ''))
        if run_id<=24:
            continue
        # --- Directory setup ---
        combo_name = f"Run_{run_id}_LR{lr:.6f}_BS{bs}"
        run_dir    = f"{SESSION_DIR}/{combo_name}"
        stage1_dir = f"{run_dir}/stage1"
        stage2_dir = f"{run_dir}/stage2"
        os.makedirs(stage1_dir, exist_ok=True)
        os.makedirs(stage2_dir, exist_ok=True)

        # FIX 6: write full run config BEFORE training starts
        # (survives even if the run crashes mid-way)
        run_config = {
            "run_id":             run_id,
            "model":              model_name,
            "rag_type":           rag_type,
            "requested_method":   requested_method,
            "effective_method":   effective_method,
            "learning_rate":      lr,
            "batch_size":         bs,
            "predicted_val_acc":  float(predicted_val_acc)  if predicted_val_acc != '' else None,
            "predicted_val_loss": float(predicted_val_loss) if predicted_val_loss != '' else None,
            "logic":              logic_text,
            "unfreeze_pct":       UNFREEZE_PCT,
            "epochs_stage1":      EPOCHS_STAGE1,
            "epochs_stage2":      EPOCHS_STAGE2,
            "session_time":       SESSION_TIME,
        }
        with open(f"{run_dir}/run_config.json", "w") as f:
            json.dump(run_config, f, indent=2)

        print(f"[{run_id}/{len(df_hpm)}] LR={lr:.6f} | BS={bs} | "
              f"{model_name} | {rag_type} | {effective_method} ...",
              end=" ", flush=True)
        t0 = time.time()

        train_ds, val_ds, test_ds, cw, n_cls, class_names = load_all_splits(DATASET_DIR, bs)

        try:
            # --- Stage 1: frozen backbone ---
            # FIX 1: stage1 uses its own checkpoint path
            s1_ckpt = f"{stage1_dir}/best_weights_s1.ckpt"
            model, base_model = build_model(n_cls, lr)
            h1 = model.fit(
                train_ds, validation_data=val_ds,
                epochs=EPOCHS_STAGE1, class_weight=cw,
                callbacks=get_callbacks(s1_ckpt, f"{stage1_dir}/epoch_logs.csv"),
                verbose=2
            )
            s1_actual = len(h1.history['accuracy'])
            model = load_best_weights(model, s1_ckpt)  # restore Stage1 best from disk

            # --- Stage 2: partial unfreeze ---
            # FIX 1: stage2 uses its own checkpoint path (does not overwrite stage1)
            s2_ckpt = f"{stage2_dir}/best_weights_s2.ckpt"
            model = unfreeze_top(model, base_model, lr)
            h2 = model.fit(
                train_ds, validation_data=val_ds,
                epochs=EPOCHS_STAGE2, class_weight=cw,
                callbacks=get_callbacks(s2_ckpt, f"{stage2_dir}/epoch_logs.csv"),
                verbose=2
            )
            model = load_best_weights(model, s2_ckpt)  # restore Stage2 best from disk

            # --- Evaluate & plot ---
            metrics = evaluate_and_plot(
                model, test_ds, class_names, run_dir,
                run_id, lr, bs, h1, h2, s1_actual
            )
            status = "SUCCESS"
            elapsed = round(time.time() - t0)
            print(f"✅  {elapsed}s  |  Test Acc: {metrics['test_accuracy']}  "
                  f"|  Macro F1: {metrics['macro_f1']}")

        except tf.errors.ResourceExhaustedError:
            metrics = {k: "OOM" for k in [
                "test_accuracy", "test_loss", "macro_f1",
                "weighted_f1",   "best_val_loss", "best_val_acc"
            ]}
            status  = "OOM Failed"
            elapsed = round(time.time() - t0)
            print(f"🚨 OOM — skipped. ({elapsed}s)")

        # --- Build summary row (all metadata + results) ---
        summary_row = {
            # Identification
            "ID":                   run_id,
            "Model":                model_name,
            "RAG_Type":             rag_type,
            "Requested_Method":     requested_method,   # FIX 4
            "Effective_Method":     effective_method,   # FIX 4
            # Hyperparameters
            "learning_rate":        lr,
            "batch_size":           bs,
            # LLM's own prediction (needed for Spearman analysis)
            "predicted_val_accuracy":  predicted_val_acc,    # FIX 3
            "predicted_val_loss":      predicted_val_loss,   # FIX 3
            # Run outcome
            "status":               status,
            "test_accuracy":        metrics["test_accuracy"],
            "test_loss":            metrics["test_loss"],
            "best_val_accuracy":    metrics["best_val_acc"],
            "best_val_loss":        metrics["best_val_loss"],
            "macro_f1":             metrics["macro_f1"],
            "weighted_f1":          metrics["weighted_f1"],
            "training_time_s":      elapsed,
            "timestamp":            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            # Logic text for traceability  FIX 5
            "logic":                logic_text,
        }

        summary_rows.append(summary_row)

        # Save full summary CSV after every run (resume safety)
        pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

        # FIX 7: also append to live session log
        append_session_log(summary_row)

        # --- Memory cleanup ---
        del train_ds, val_ds, test_ds
        if 'model'      in locals(): del model
        if 'base_model' in locals(): del base_model
        aggressive_cleanup()

    print(f"\n✅ All runs complete → {SESSION_DIR}")
    print(f"   Summary CSV  : {SUMMARY_CSV}")
    print(f"   Session log  : {SESSION_LOG_CSV}")


if __name__ == "__main__":
    run_all_experiments()


🚀 Starting 36 LLM-HPO runs  |  Session: 2026-03-16_11-13-05
   UNFREEZE_PCT=0.3 | amsgrad=OFF | ReduceLR=OFF

[25/36] LR=0.000800 | BS=192 | gemini-3.1-pro-preview | no_context |  ... Found 11215 files belonging to 9 classes.
Found 3202 files belonging to 9 classes.
Found 1610 files belonging to 9 classes.
Epoch 1/3
59/59 - 181s - loss: 1.0722 - accuracy: 0.3234 - val_loss: 1.7131 - val_accuracy: 0.3770 - 181s/epoch - 3s/step
Epoch 2/3
59/59 - 197s - loss: 0.8298 - accuracy: 0.5499 - val_loss: 1.4750 - val_accuracy: 0.4916 - 197s/epoch - 3s/step
Epoch 3/3
59/59 - 199s - loss: 0.7250 - accuracy: 0.6189 - val_loss: 1.2866 - val_accuracy: 0.5662 - 199s/epoch - 3s/step
Epoch 1/5
59/59 - 227s - loss: 0.3425 - accuracy: 0.7942 - val_loss: 0.2824 - val_accuracy: 0.9116 - 227s/epoch - 4s/step
Epoch 2/5
59/59 - 206s - loss: 0.1684 - accuracy: 0.8903 - val_loss: 0.2815 - val_accuracy: 0.8998 - 206s/epoch - 3s/step
Epoch 3/5
59/59 - 176s - loss: 0.1078 - accuracy: 0.9312 - val_loss: 0.1457 - val

Epoch 3/5
701/701 - 360s - loss: 0.1271 - accuracy: 0.9612 - val_loss: 0.1313 - val_accuracy: 0.9613 - 360s/epoch - 514ms/step
Epoch 4/5
701/701 - 386s - loss: 0.0965 - accuracy: 0.9705 - val_loss: 0.1569 - val_accuracy: 0.9535 - 386s/epoch - 550ms/step
✅  2658s  |  Test Acc: 0.9627  |  Macro F1: 0.8601
[32/36] LR=0.001600 | BS=16 | gpt-5.2-pro | best_values |  ... Found 11215 files belonging to 9 classes.
Found 3202 files belonging to 9 classes.
Found 1610 files belonging to 9 classes.
Epoch 1/3


KeyboardInterrupt: 